In [ ]:
# 1. Kết nối với Drive
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


In [ ]:
# 1. CÀI ĐẶT MÔI TRƯỜNG VÀ THƯ VIỆN (Chỉ chạy 1 lần)
# Thêm -qq và -q để ẩn bớt log cài đặt cho gọn màn hình
!apt-get install openjdk-11-jdk-headless -qq -y
!pip install pyspark==3.5.0 findspark notebook plotly pandas matplotlib graphviz -q

# Ghi nhận Jupyter kernel
!python -m ipykernel install --user --name pyspark-env --display-name "Python (PySpark)"

# 2. IMPORT THƯ VIỆN & THIẾT LẬP BIẾN MÔI TRƯỜNG
import os
import sys
import findspark
import pyspark
import pandas as pd
import matplotlib

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/usr/local/lib/python3.12/dist-packages/pyspark"

# Khởi tạo findspark
findspark.init()

# 3. KIỂM TRA VÀ IN VERSION (Theo yêu cầu)
print("="*30)
print("THÔNG TIN MÔI TRƯỜNG & PHIÊN BẢN")
print("="*30)
print(f"Python version: {sys.version.split(' ')[0]}")
!java -version
print(f"PySpark version: {pyspark.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"JAVA_HOME = {os.environ.get('JAVA_HOME')}")
print(f"SPARK_HOME = {os.environ.get('SPARK_HOME')}")
print("="*30)

# 4. KHỞI TẠO SPARK SESSION & TEST
from pyspark.sql import SparkSession

# Gộp cấu hình offHeap vào SparkSession chính thức
spark = SparkSession.builder \
    .appName("Olist_Machine_Learning") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "10g") \
    .getOrCreate()

print("\nĐã khởi tạo SparkSession thành công!")

# Chạy test DataFrame để đảm bảo Spark đang hoạt động tốt
df = spark.createDataFrame([(1, "hello"), (2, "world")], ["id", "text"])
df.show()

In [ ]:
data_path = "/content/gdrive/MyDrive/BIGDATA/Dataset cuối kỳ/Dataset cuối kỳ/cleaned_master_data.parquet"

# Lúc này biến 'spark' đã tồn tại nên sẽ không bị lỗi nữa
df_master = spark.read.parquet(data_path)

# Kiểm tra xem dữ liệu đã lên chưa
df_master.show(5)

In [ ]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

# 1. Đếm số lần mua hàng của mỗi khách hàng cho từng sản phẩm
print("Đang tổng hợp dữ liệu tương tác (Implicit Feedback)...")
interaction_df = df_master.groupBy("customer_unique_id", "product_id") \
                          .agg(F.count("*").alias("rating"))

# 2. Mã hóa chuỗi ID dài thành số nguyên liên tục (bắt buộc cho ALS)
user_indexer = StringIndexer(inputCol="customer_unique_id", outputCol="user_index", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_index", handleInvalid="skip")

# Áp dụng mã hóa
df_indexed = user_indexer.fit(interaction_df).transform(interaction_df)
df_indexed = item_indexer.fit(df_indexed).transform(df_indexed)

# 3. Ép kiểu dữ liệu (ALS yêu cầu index là dạng integer, rating là float)
df_final = df_indexed.withColumn("user_index", F.col("user_index").cast("integer")) \
                     .withColumn("item_index", F.col("item_index").cast("integer")) \
                     .withColumn("rating", F.col("rating").cast("float"))

print("Dữ liệu sau khi chuẩn bị xong:")
df_final.select("user_index", "item_index", "rating").show(5)

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# 1. Chia tập Train (80%) và Test (20%)
train_data, test_data = df_final.randomSplit([0.8, 0.2], seed=42)

# 2. Khởi tạo thuật toán ALS (Bật implicitPrefs và coldStartStrategy)
als = ALS(
    userCol="user_index",
    itemCol="item_index",
    ratingCol="rating",
    implicitPrefs=True,       # ĐÁNH DẤU LÀ DỮ LIỆU NGẦM
    coldStartStrategy="drop", # Tránh lỗi cho các user không có trong tập Train
    nonnegative=True  # Không cho phép dự đoán âm

)

# 3. Thiết lập GridSearch (Tuning siêu tham số)
# Lưu ý: Tôi để grid vừa phải để chạy không bị quá lâu. Bạn có thể mở rộng thêm nếu có máy chủ mạnh.
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [10, 20]) \
    .addGrid(als.regParam, [0.01, 0.1]) \
    .addGrid(als.alpha, [1.0, 10.0]) \
    .build()

# 4. Khởi tạo công cụ đánh giá (dùng RMSE)
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

# 5. Khởi tạo TrainValidationSplit để tự động tìm tham số
tvs = TrainValidationSplit(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8 # Trong lúc tuning, chia 80% train để học, 20% validation để test nội bộ
)

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col

# 1. ĐẢM BẢO SPARK SESSION ĐANG CHẠY
spark = SparkSession.builder.getOrCreate()
print("✅ Spark Session đã sẵn sàng!")

# =====================================================================
# BƯỚC 1: HUẤN LUYỆN (TRAINING)
# =====================================================================
print("🚀 Bắt đầu quá trình Tuning & Training... (Vui lòng đợi)")
start_time = time.time()

# Chạy tuning để tìm mô hình tốt nhất
tvs_model = tvs.fit(train_data)
best_als_model = tvs_model.bestModel

end_time = time.time()
print(f"✔️ Hoàn tất sau {round((end_time - start_time)/60, 2)} phút!")

# In thông số tốt nhất
print(f"📍 Rank tối ưu: {best_als_model.rank}")
# Đánh giá trên tập Test
predictions = best_als_model.transform(test_data)
rmse = evaluator.evaluate(predictions)
print(f"📊 RMSE trên tập Test: {rmse:.4f}")
print(f"RegParam: {best_als_model._java_obj.parent().getRegParam()}")
print(f"Alpha: {best_als_model._java_obj.parent().getAlpha()}")

# =====================================================================
# BƯỚC 2: TẠO GỢI Ý VÀ LÀM ĐẸP DỮ LIỆU
# =====================================================================
print("\n🔍 Đang trích xuất Top 10 gợi ý và thêm tên danh mục...")

# 1. Lấy gợi ý thô
user_recs = best_als_model.recommendForAllUsers(10)

# 2. Bung mảng (Explode)
flat_recs = user_recs.withColumn("rec", explode("recommendations")) \
                     .select(
                         "user_index",
                         col("rec.item_index").alias("item_index"),
                         col("rec.rating").alias("preference_score")
                     )

# 3. Tạo bảng map ID
user_map = df_final.select("user_index", "customer_unique_id").distinct()
item_map = df_final.select("item_index", "product_id").distinct()
product_cat_map = df_master.select("product_id", "product_category_name").distinct()

# 4. Join tất cả lại thành bảng hoàn chỉnh
readable_recs = flat_recs.join(user_map, on="user_index", how="inner") \
                         .join(item_map, on="item_index", how="inner") \
                         .join(product_cat_map, on="product_id", how="left")

# =====================================================================
# BƯỚC 3: IN KẾT QUẢ ĐẸP CHO 3 USER MẪU
# =====================================================================
# In tiêu đề báo cáo theo đúng format yêu cầu
print("\n==========================================================")
print(f"[RMSE trên tập test: {rmse:.4f}. Trình bày top 10 recommendations cho 3 khách hàng mẫu.]")
print("==========================================================")

# Lấy tự động 3 user_index bất kỳ có trong danh sách gợi ý
sample_users_rows = readable_recs.select("user_index").distinct().limit(3).collect()
sample_users = [row['user_index'] for row in sample_users_rows]

# Vòng lặp in kết quả cho từng User
for i, u_index in enumerate(sample_users, 1):
    print(f"\n--- KHÁCH HÀNG MẪU SỐ {i} (USER INDEX: {u_index}) ---")
    readable_recs.filter(col("user_index") == u_index) \
                 .select("customer_unique_id", "product_category_name", "product_id", "preference_score") \
                 .orderBy(col("preference_score").desc()) \
                 .show(10, truncate=False)

✅ Spark Session đã sẵn sàng!
🚀 Bắt đầu quá trình Tuning & Training... (Vui lòng đợi)
✔️ Hoàn tất sau 55.63 phút!
📍 Rank tối ưu: 10
📊 RMSE trên tập Test: 1.2763
RegParam: 0.1
Alpha: 10.0

🔍 Đang trích xuất Top 10 gợi ý và thêm tên danh mục...

[RMSE trên tập test: 1.2763. Trình bày top 10 recommendations cho 3 khách hàng mẫu.]

--- KHÁCH HÀNG MẪU SỐ 1 (USER INDEX: 148) ---
+--------------------------------+----------------------+--------------------------------+----------------+
|customer_unique_id              |product_category_name |product_id                      |preference_score|
+--------------------------------+----------------------+--------------------------------+----------------+
|b471f6779ed58f5dbcf786cf9842067a|ferramentas_jardim    |368c6c730842d78016ad823897a372db|0.34031072      |
|b471f6779ed58f5dbcf786cf9842067a|relogios_presentes    |a62e25e09e05e6faf31d90c6ec1aa3d1|0.27764568      |
|b471f6779ed58f5dbcf786cf9842067a|beleza_saude          |437c05a395e9e47f9762e677a706

In [ ]:
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS

# ==========================================================
# BƯỚC 1: ĐỊNH NGHĨA CÁC THÀNH PHẦN (STAGES)
# ==========================================================

# 1.1. Mã hóa customer_unique_id -> user_index
user_indexer = StringIndexer(
    inputCol="customer_unique_id",
    outputCol="user_index",
    handleInvalid="skip"
)

# 1.2. Mã hóa product_id -> item_index
item_indexer = StringIndexer(
    inputCol="product_id",
    outputCol="item_index",
    handleInvalid="skip"
)

# 1.3. Khởi tạo mô hình ALS với các tham số TỐT NHẤT bạn đã tìm thấy
# Thay rank, regParam, alpha bằng các số in ra từ bước Tuning của bạn
best_rank = best_als_model.rank
best_reg = best_als_model._java_obj.parent().getRegParam()
best_alpha = best_als_model._java_obj.parent().getAlpha()

als = ALS(
    userCol="user_index",
    itemCol="item_index",
    ratingCol="rating",
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    rank=best_rank,
    regParam=best_reg,
    alpha=best_alpha
)

# ==========================================================
# BƯỚC 2: XÂY DỰNG VÀ HUẤN LUYỆN PIPELINE
# ==========================================================
# Gom tất cả vào 1 luồng duy nhất
rec_pipeline = Pipeline(stages=[user_indexer, item_indexer, als])

print("🚀 Đang huấn luyện End-to-End Pipeline...")
# Lưu ý: fit trên bảng interaction_df (bảng gốc chưa index)
# vì Pipeline sẽ tự làm bước Indexing
final_pipeline_model = rec_pipeline.fit(interaction_df)

# ==========================================================
# BƯỚC 3: LƯU PIPELINE ĐỂ DÙNG CHO GRADIO
# ==========================================================
save_path = "/content/gdrive/MyDrive/BIGDATA/gradio_recommendation_als_pipeline"
final_pipeline_model.write().overwrite().save(save_path)

print(f"✅ Đã lưu Pipeline thành công tại: {save_path}")

🚀 Đang huấn luyện End-to-End Pipeline...
✅ Đã lưu Pipeline thành công tại: /content/gdrive/MyDrive/BIGDATA/gradio_recommendation_als_pipeline
